# ID Forensics — Stage 1 corner training on Colab Pro

**Before opening this notebook:**
1. On your work PC run `pack_for_home.py` and upload `id_forensics_home_data.zip` to Google Drive.
2. In VS Code: install the **Google Colab** extension, open this file, select kernel **Colab → New Colab Server → GPU (T4)**.
3. Edit `ZIP_PATH` in the next cell if your zip is not in Drive root.

In [ ]:
# --- EDIT THESE ---
REPO_URL = "https://github.com/YOUR_USER/id-forensics-model.git"  # your GitHub repo
ZIP_PATH = "/content/drive/MyDrive/id_forensics_home_data.zip"     # path on Google Drive
EPOCHS = 50
BATCH = 16

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import zipfile
from pathlib import Path

if not Path(ZIP_PATH).is_file():
    raise FileNotFoundError(f"Zip not found: {ZIP_PATH}")

os.chdir("/content")
if not Path("id-forensics-model").is_dir():
    !git clone {REPO_URL} id-forensics-model

os.chdir("/content/id-forensics-model")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(".")
print("Repo ready:", os.getcwd())

In [ ]:
!pip install -q ultralytics opencv-python-headless timm python-dotenv
!python scripts/verify_home_setup.py

In [ ]:
# Delete stale YOLO label caches (safe after format changes)
from pathlib import Path
for cache in Path("data/yolo/corners/labels").rglob("*.cache"):
    cache.unlink()
    print("deleted", cache)

In [ ]:
!python scripts/training/train_stage1_corners.py --device 0 --epochs {EPOCHS} --batch {BATCH}

In [ ]:
import shutil
from pathlib import Path

best = Path("models/stage1_corners/weights/best.pt")
if not best.is_file():
    raise FileNotFoundError(best)

out = Path("/content/drive/MyDrive/id_forensics_stage1_best.pt")
shutil.copy2(best, out)
print(f"Saved to Drive: {out}")
print(f"Size: {out.stat().st_size / 1e6:.1f} MB")